In [108]:
import pandas as pd
import streamlit as st
import matplotlib.pyplot as plt
import numpy as np

csv_customers_dataset = pd.read_csv('./streamlit_resources/customers_dataset.csv')

csv_orders_dataset = pd.read_csv('./streamlit_resources/orders_dataset.csv')

csv_reviews_dataset = pd.read_csv('./streamlit_resources/order_reviews_dataset.csv')

# 1. Clientes por estado y ciudad

Representa una clasificación del número de clientes por estado. Crea una tabla en la que se muestren:

- Estado
- Ciudad
- Número de clientes por ciudad

In [109]:
''' Comprobación de nulos para limpiar los datos
csv_customers_dataset['customer_zip_code_prefix'].isna().any()
csv_customers_dataset['customer_city'].isna().any()
csv_customers_dataset['customer_state'].isna().any() 
No se han encontrado nulos en estas columnas'''

customers_by_city_state = csv_customers_dataset.groupby(
    ['customer_state', 'customer_city']
)['customer_id'].nunique().reset_index()

### Tanto la tabla como los gráficos deberán ser dinámicos respecto a la fecha para permitir el análisis temporal de la evolución de clientes.

In [110]:
'''
Para realizar el filtro tomamos la fecha de compra desde orders_dataset, para compararlo con la fecha actual
'''

# Comprobación de nulos en fecha antes del casting
csv_orders_dataset['order_purchase_timestamp'].isna().any()

# Casting a fecha
csv_orders_dataset['order_purchase_timestamp'] = csv_orders_dataset['order_purchase_timestamp'].apply(pd.to_datetime)

csv_orders_dataset_format = csv_orders_dataset['order_purchase_timestamp'].min()

merge_data_to_compare = pd.merge(csv_orders_dataset,csv_customers_dataset, on='customer_id')

delivered_orders = merge_data_to_compare[merge_data_to_compare['order_status'] == 'delivered']

customers_by_city_state = csv_customers_dataset.groupby(
    ['customer_state', 'customer_city']
)['customer_unique_id'].nunique().reset_index()

dfcustomer = customers_by_city_state.sort_values('customer_unique_id',ascending=False)

dfcustomer

,customer_state,customer_city,customer_unique_id
4176,SP,sao paulo,14984
2788,RJ,rio de janeiro,6620
1062,MG,belo horizonte,2672
601,DF,brasilia,2069
2406,PR,curitiba,1465
...,...,...,...
4298,TO,pequizeiro,1
4297,TO,peixe,1
4292,TO,novo jardim,1
4291,TO,novo alegre,1


# 2. Pedidos por ciudad
## A la tabla anterior añade las siguientes columnas:

Número de pedidos
Porcentaje que representan respecto al total de pedidos
Además, representa el ratio de pedidos por cliente, utilizando el tipo de gráfico que consideres más adecuado.

In [111]:
orders_by_city = merge_data_to_compare.groupby(
    ['customer_state', 'customer_city']
)['order_id'].count().reset_index()

total_orders = merge_data_to_compare['order_id'].nunique()

orders_by_city['order_percentage'] = (
    orders_by_city['order_id'] / total_orders
) * 100

customers_by_city = merge_data_to_compare.groupby(
    ['customer_state', 'customer_city']
)['customer_unique_id'].nunique().reset_index()

city_stats = pd.merge(
    customers_by_city,
    orders_by_city,
    on=['customer_state', 'customer_city']
)

# customer_unique_id identifica al cliente REAL (siempre el mismo aunque haga varios pedidos)
# customer_id cambia en cada pedido, por eso sirve para contar pedidos, no clientes
# order_id es el identificador del pedido
city_stats['orders_per_customer'] = (
    city_stats['order_id'] / city_stats['customer_unique_id']
).round(2)

city_stats = city_stats.rename(columns={
    'order_id': 'Número de pedidos',
    'order_percentage': 'Porcentaje de pedidos',
    'customer_unique_id': 'Número de clientes',
    'orders_per_customer':'Ratio de pedidos por cliente'
})

city_stats.sort_values('Número de clientes', ascending=False)

,customer_state,customer_city,Número de clientes,Número de pedidos,Porcentaje de pedidos,Ratio de pedidos por cliente
4176,SP,sao paulo,14984,15540,15.627357,1.04
2788,RJ,rio de janeiro,6620,6882,6.920687,1.04
1062,MG,belo horizonte,2672,2773,2.788588,1.04
601,DF,brasilia,2069,2131,2.142979,1.03
2406,PR,curitiba,1465,1521,1.529550,1.04
...,...,...,...,...,...,...
4298,TO,pequizeiro,1,1,0.001006,1.00
4297,TO,peixe,1,1,0.001006,1.00
4292,TO,novo jardim,1,1,0.001006,1.00
4291,TO,novo alegre,1,1,0.001006,1.00


# 3. Análisis de retrasos en pedidos
### Calcula y representa:

Número de pedidos que llegan tarde por ciudad
Porcentaje de pedidos retrasados respecto al total de pedidos de la ciudad
Tiempo medio de retraso en días
Además, al representar esta información, el dashboard deberá incluir un autodiagnóstico que indique la razón más probable del problema.

In [112]:
pd.reset_option("all")

# Convertir columnas a fecha
delivered_orders['order_delivered_customer_date'] = pd.to_datetime(delivered_orders['order_delivered_customer_date'])
delivered_orders['order_estimated_delivery_date'] = pd.to_datetime(delivered_orders['order_estimated_delivery_date'])

# Días de retraso
delivered_orders['delay_days'] = (delivered_orders['order_delivered_customer_date'] - delivered_orders['order_estimated_delivery_date']).dt.days

# Pedidos retrasados (por diferencia de días)
delivered_orders['is_late'] = delivered_orders['delay_days'] > 0

# Número de pedidos retrasados por ciudad (agrupado por estados, ya que hay ciudades que coinciden en varios estados)
late_by_city = delivered_orders.groupby(
    ['customer_state', 'customer_city']
)['is_late'].sum().reset_index()

# Total de pedidos por ciudad (agrupado por estados, ya que hay ciudades que coinciden en varios estados)
total_by_city = delivered_orders.groupby(
    ['customer_state', 'customer_city']
)['order_id'].nunique().reset_index()

delay_all_stats = total_by_city.merge(late_by_city, on=["customer_state","customer_city"])

delay_all_stats['Porcentaje retrasados'] = (
    delay_all_stats['is_late'] / delay_all_stats['order_id'] * 100
).round(2)

# Retraso medio por ciudad
avg_delay_by_city = delivered_orders[delivered_orders['is_late']].groupby(
    ['customer_state', 'customer_city']
)['delay_days'].mean().reset_index()

avg_delay_by_city = avg_delay_by_city.rename(columns={
    'delay_days': 'Retraso medio en días'
})

delay_all_stats = delay_all_stats.merge(avg_delay_by_city, on=['customer_state','customer_city'])

delay_all_stats = delay_all_stats.rename(columns={
    'order_id': 'Total pedidos',
    'is_late': 'Pedidos retrasados',
})

delay_all_stats.sort_values(by='Pedidos retrasados',ascending=False)


C:\Users\jgarcia\AppData\Local\Temp\ipykernel_19760\2929706914.py:1: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  pd.reset_option("all")
C:\Users\jgarcia\AppData\Local\Temp\ipykernel_19760\2929706914.py:1: Pandas4Warning: The 'mode.copy_on_write' option is deprecated. Copy-on-Write can no longer be disabled (it is always enabled with pandas >= 3.0), and setting the option has no impact. This option will be removed in pandas 4.0.
  pd.reset_option("all")


,customer_state,customer_city,Total pedidos,Pedidos retrasados,Porcentaje retrasados,Retraso medio en días
1160,SP,sao paulo,15045,715,4.75,7.534266
746,RJ,rio de janeiro,6601,706,10.70,13.339943
103,BA,salvador,1188,174,14.65,10.810345
314,MG,belo horizonte,2697,137,5.08,7.401460
835,RS,porto alegre,1342,136,10.13,10.551471
...,...,...,...,...,...,...
1190,SP,vinhedo,90,1,1.11,4.000000
1189,SP,vera cruz,9,1,11.11,9.000000
1187,SP,vargem grande paulista,43,1,2.33,8.000000
16,AL,sao luis do quitunde,2,1,50.00,15.000000


# 4. Reviews y satisfacción del cliente
Calcula y representa:

- Número de reviews por estado
- Score medio de las reviews en cada estado
- Para este cálculo, se deberán excluir los pedidos con retraso, ya que se entiende que la valoración negativa podría deberse principalmente al retraso en la entrega del producto.

In [113]:
reviews_products_delivered = delivered_orders.merge(csv_reviews_dataset[['order_id', 'review_score']], on='order_id')

# Comprobación de que no exista review sin puntuación
reviews_products_delivered['review_score'].isna().sum()

on_time_reviews = reviews_products_delivered[
    (reviews_products_delivered['is_late'] == False) &
    (reviews_products_delivered['review_score'].notna())
]

reviews_by_state = on_time_reviews.groupby('customer_state')['review_score'].count().reset_index(name='num_reviews')

avg_score_by_state = on_time_reviews.groupby('customer_state')['review_score'].mean().round(2).reset_index(name='score_medio')

reviews_stats = reviews_by_state.merge(avg_score_by_state, on='customer_state').sort_values(by='score_medio', ascending=False)

reviews_stats

,customer_state,num_reviews,score_medio
19,RN,430,4.39
11,MS,642,4.37
25,SP,38695,4.32
22,RS,5039,4.31
17,PR,4724,4.30
15,PE,1441,4.30
26,TO,246,4.30
1,AL,315,4.30
23,SC,3247,4.29
2,AM,141,4.28
